# Anomaly Detection — Isolation Forest

Detect **fall events** and **unexpected movement** without any pre-defined
thresholds — purely from learning what _normal riding_ looks like.

| | Rule-based | Isolation Forest |
|---|---|---|
| Training data needed | None | Normal-riding windows |
| Threshold | Hard-coded 2.0 rad/s | Data-driven contamination % |
| Multi-axis | Magnitude only | All 39 features |
| Adapts to new conditions | No | Retrain on new data |

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent.parent
sys.path.insert(0, str(ROOT / 'ml'))
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve, roc_auc_score

from scooter_imu.data.loader       import load_raw
from scooter_imu.data.preprocessor import preprocess
from scooter_imu.algorithms.fall_detection import FallDetector

from features  import extract_windows
from labeler   import label_timestamps, CLASS_NAMES, CLASS_TO_IDX
from anomaly   import IMUAnomalyDetector

df = preprocess(load_raw(ROOT / 'data/raw/ScooterIMUData.csv'))
t0 = df['timestamp'].iloc[0]
print(f'Loaded {len(df):,} rows')

Loaded 30,000 rows


## 1. Extract Windows & Define Normal Set

In [2]:
X, mid_ts = extract_windows(df, window_s=1.0, step_s=0.5)
y = label_timestamps(mid_ts, t0)

# Anomalous classes — excluded from training
ANOMALY_CLASSES = {'fall', 'parked'}
ANOMALY_IDX     = {CLASS_TO_IDX[c] for c in ANOMALY_CLASSES}

normal_mask  = np.array([i >= 0 and i not in ANOMALY_IDX for i in y])
fall_mask    = y == CLASS_TO_IDX['fall']
X_normal     = X[normal_mask]

print(f'Total windows  : {len(X)}')
print(f'Normal windows : {normal_mask.sum()}')
print(f'Fall windows   : {fall_mask.sum()}')

Total windows  : 1199
Normal windows : 1020
Fall windows   : 20


## 2. Train Isolation Forest

In [3]:
det = IMUAnomalyDetector(contamination=0.05, n_estimators=200)
det.fit(X_normal)

scores = det.anomaly_score(X)
print(f'Anomaly score range: [{scores.min():.3f}, {scores.max():.3f}]')
print(f'Fall window scores : mean={scores[fall_mask].mean():.3f}  '
      f'max={scores[fall_mask].max():.3f}')
print(f'Normal window scores: mean={scores[normal_mask].mean():.3f}')

Anomaly score range: [0.000, 1.000]
Fall window scores : mean=0.359  max=1.000
Normal window scores: mean=0.230


## 3. Anomaly Score Over the Entire Trip

In [4]:
# Time axis in seconds from trip start
t_sec = (np.asarray(mid_ts, dtype='datetime64[ns]') - np.datetime64(t0, 'ns')) / np.timedelta64(1, 's')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=t_sec, y=scores,
    mode='lines', line=dict(color='#2E86AB', width=1),
    name='Anomaly Score',
))
fig.add_trace(go.Scatter(
    x=t_sec[fall_mask], y=scores[fall_mask],
    mode='markers', marker=dict(color='#E84855', size=8, symbol='x'),
    name='Fall Windows',
))

# Add threshold line
threshold = 0.65
fig.add_hline(y=threshold, line_dash='dash', line_color='orange',
              annotation_text=f'Threshold ({threshold})')

# Shade known anomaly regions
for name, cls, color in [('Fall Event', 'fall', 'rgba(232,72,85,0.15)'),
                          ('Parked',     'parked','rgba(100,100,100,0.1)')]:
    mask = y == CLASS_TO_IDX[cls]
    if mask.sum() > 0:
        x_start = t_sec[mask].min()
        x_end   = t_sec[mask].max()
        fig.add_vrect(x0=x_start, x1=x_end, fillcolor=color,
                      line_width=0, annotation_text=name,
                      annotation_position='top left')

fig.update_layout(
    title='Isolation Forest Anomaly Score Over Trip',
    xaxis_title='Trip time (s)', yaxis_title='Anomaly Score',
    height=400, template='plotly_white', showlegend=True,
)
fig.show()


## 4. Precision-Recall vs Rule-Based FallDetector

In [5]:
# Ground truth: 1 = anomaly (fall OR parked movement)
y_true = np.array([1 if i in ANOMALY_IDX else 0 for i in y])

# Isolation Forest PR curve
prec, rec, thresh = precision_recall_curve(y_true, scores)

# Rule-based: per-window fraction of fall_event flags
rb = FallDetector().predict(df)
dt_med = df['dt'].median()
win_samps = int(1.0 / dt_med)
rb_score = pd.Series(rb['fall_event'].astype(float)).rolling(win_samps, center=True).mean().to_numpy()
# Sample at window centres
ts_arr  = df['timestamp'].to_numpy()
rb_w    = np.interp(
    (np.asarray(mid_ts, dtype='datetime64[ns]') - np.asarray(ts_arr[:1], dtype='datetime64[ns]')[0]) / np.timedelta64(1, 's'),
    (ts_arr - ts_arr[0]) / np.timedelta64(1, 's'),
    rb_score,
)
prec_rb, rec_rb, _ = precision_recall_curve(y_true, rb_w)

fig = go.Figure()
fig.add_trace(go.Scatter(x=rec,    y=prec,    mode='lines',
                         name=f'Isolation Forest  (AUC={roc_auc_score(y_true, scores):.3f})',
                         line=dict(color='#2E86AB')))
fig.add_trace(go.Scatter(x=rec_rb, y=prec_rb, mode='lines',
                         name=f'Rule-Based FallDetector',
                         line=dict(color='#E84855', dash='dash')))
fig.update_layout(
    title='Precision-Recall Curve — Anomaly Detection',
    xaxis_title='Recall', yaxis_title='Precision',
    height=400, template='plotly_white',
)
fig.show()


## 5. Save Model

In [6]:
det.save(ROOT / 'ml/models/anomaly_iforest.joblib')
print('Model saved. Reload with:  IMUAnomalyDetector.load()')

IMUAnomalyDetector saved → /Users/pulkit/Desktop/test/Automatic-Indicator-using-Gyroscope-data/ml/models/anomaly_iforest.joblib
Model saved. Reload with:  IMUAnomalyDetector.load()
